# Experiment 2.1a: Conjugation Single-Step Exact Covariance

## Pair identity and scope

This notebook is the reporting companion to `run_experiment.py` for
`Exp 2.1a_Conjugation_Single-Step_Exact_Covariance`. It **does not**
reimplement the experiment core. Instead, it imports the script, runs the
script's `run_experiment()` function with the default toy configuration, and
then presents the returned results in a more explicit academic format.

The question under test is the sampled numerical identity

$$
\mathrm{muon\_step}(RWS^	op,\;RGS^	op)\;=\;R\,\mathrm{muon\_step}(W,G)\,S^	op
$$

for **orthogonal** conjugators $R \in O(m)$ and $S \in O(n)$, in the
**zero-momentum single-step** setting, using the **finite Newton-Schulz
iterate actually implemented in the script**.

This is a **sampled numerical verification**, not a universal proof over the
full orthogonal group. It also does not test momentum, long training runs,
or convergence of the finite Newton-Schulz iterate to the exact polar factor.

## What this notebook reports

1. notebook-safe import/path handling for the paired script
2. reproducibility and runtime/config logging
3. a finite-iteration Newton-Schulz sanity check
4. orthogonal-trial summary tables and error-distribution figures
5. orthogonal vs non-orthogonal control comparison
6. Newton-Schulz iteration-count sensitivity
7. explicit H1-H4 outcomes returned by the script
8. a calibrated conclusion tied only to the observed sampled evidence

In [ ]:
from pathlib import Path
import importlib.util
import time

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display, HTML, Markdown
except Exception:
    display = None
    HTML = None
    Markdown = None

EXPERIMENT_RELATIVE_PATH = Path(
    "experiments/Tier_2_Symmetry_Reparametrization_Tests/"
    "Exp_2.1_Conjugation_Covariance/"
    "2.1a_Conjugation_Single-Step_Exact_Covariance/"
    "run_experiment.py"
)

def locate_script():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        candidate = root / EXPERIMENT_RELATIVE_PATH
        if candidate.exists():
            return root.resolve(), candidate.resolve()
    raise FileNotFoundError(
        f"Could not locate {EXPERIMENT_RELATIVE_PATH} from cwd={Path.cwd()}"
    )

REPO_ROOT, SCRIPT_PATH = locate_script()
SPEC = importlib.util.spec_from_file_location("exp_2_1a_run_experiment", SCRIPT_PATH)
if SPEC is None or SPEC.loader is None:
    raise ImportError(f"Could not load module spec from {SCRIPT_PATH}")

exp21a = importlib.util.module_from_spec(SPEC)
SPEC.loader.exec_module(exp21a)

print("Notebook/script linkage established")
print(f"  Repository root : {REPO_ROOT}")
print(f"  Script path     : {SCRIPT_PATH}")
print(f"  Imported module : {exp21a.__name__}")

## Reproducibility, runtime, and default configuration

The script remains the authoritative computation engine. The notebook simply
calls `run_experiment()` and records the returned configuration, seeds, and
runtime. This keeps notebook behavior aligned with script behavior.

In [ ]:
notebook_start = time.perf_counter()
results = exp21a.run_experiment()
notebook_runtime = time.perf_counter() - notebook_start

config = results['config']
orthogonal_results = results['orthogonal_results']
orthogonal_aggregate = results['orthogonal_aggregate']
nonorthogonal_control = results['nonorthogonal_control']
ns_sensitivity = results['ns_sensitivity']
hypotheses = results['hypotheses']
verdict = results['verdict']

print("Experiment metadata")
print("-" * 72)
print(f"  Experiment ID                  : {results['experiment_id']}")
print(f"  Title                          : {results['title']}")
print(f"  Identity under test            : {results['identity_under_test']}")
print(f"  Scope                          : {results['scope']}")
print("  Implementation notes           :")
for key, value in results['implementation_notes'].items():
    label = key.replace('_', ' ')
    print(f"    - {label}: {value}")
print()
print("Numerical/runtime configuration")
print("-" * 72)
print(f"  Sizes                          : {config['sizes']}")
print(f"  Trials per size                : {config['n_trials']}")
print(f"  Learning rate                  : {config['lr']}")
print(f"  Default Newton-Schulz iters    : {config['ns_iters']}")
print(f"  Base RNG seed                  : {config['base_seed']}")
print(f"  Python executable              : {config['python_executable']}")
print(f"  Python version                 : {config['python_version']}")
print(f"  Script path                    : {config['script_path']}")
print(f"  Orthogonal seed policy         : {config['orthogonal_seed_policy']}")
print(f"  Non-orthogonal control seed    : {config['nonorthogonal_control_seed']}")
print(f"  NS sensitivity seeds           : {config['ns_sensitivity_seeds']}")
print(f"  dtype                          : {config['dtype']}")
print(f"  NumPy version                  : {config['numpy_version']}")
print(f"  Float64 machine epsilon        : {np.finfo(np.float64).eps:.2e}")
print()
print("Runtime")
print("-" * 72)
print(f"  Script-reported compute time   : {results['runtime_sec']:.4f} s")
print(f"  Notebook wall-clock time       : {notebook_runtime:.4f} s")
print("  Environment note               : pure NumPy, CPU, tiny matrices, no file I/O")

## Finite Newton-Schulz map: what is actually being tested

The script does **not** compute an exact polar factor. It applies the finite
Newton-Schulz map

$$
X_{k+1}=	frac{3}{2}X_k - 	frac{1}{2}X_k(X_k^	op X_k),\qquad X_0 = M / \|M\|_F
$$

for a fixed number of iterations (`NS_ITERS = 5` by default). The conjugation
equivariance claim tested here concerns this **finite iterate** and the
resulting single Muon step. Full convergence to the exact polar factor is a
different question.

In [ ]:
rng_check = np.random.RandomState(0)
M_check = rng_check.randn(4, 4)
Q_check = exp21a.newton_schulz(M_check, n_iters=config['ns_iters'])
ortho_residual = np.linalg.norm(Q_check.T @ Q_check - np.eye(4), ord='fro')

print("Finite-iteration Newton-Schulz sanity check")
print("-" * 72)
print(f"  Input Frobenius norm           : {np.linalg.norm(M_check, ord='fro'):.4f}")
print(f"  Input condition number         : {np.linalg.cond(M_check):.2f}")
print(f"  Output ||Q^T Q - I||_F         : {ortho_residual:.2e}")
print(f"  Iterations used                : {config['ns_iters']}")
print()
print("Interpretation:")
print("  A sizable orthogonality residual here is not a contradiction. It simply")
print("  shows that the finite Newton-Schulz iterate need not have fully converged")
print("  to the exact polar factor on an arbitrary random input after 5 steps.")
print("  The present experiment tests conjugation equivariance of that finite map,")
print("  not global convergence quality of Newton-Schulz.")

## Helper functions for tables and calibrated notebook output

In [ ]:
def in_ipython_frontend():
    try:
        return get_ipython() is not None
    except NameError:
        return False

def sci(x):
    return f"{x:.2e}"

def table_to_text(columns, rows):
    widths = {col: len(col) for col in columns}
    for row in rows:
        for col in columns:
            widths[col] = max(widths[col], len(str(row[col])))
    header = " | ".join(f"{col:<{widths[col]}}" for col in columns)
    sep = "-+-".join("-" * widths[col] for col in columns)
    body = [" | ".join(f"{str(row[col]):<{widths[col]}}" for col in columns) for row in rows]
    return "\n".join([header, sep, *body])

def table_to_html(columns, rows):
    header = "".join(f"<th style='padding:6px 10px;text-align:left'>{col}</th>" for col in columns)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td style='padding:6px 10px;text-align:left'>{row[col]}</td>" for col in columns)
        body_rows.append(f"<tr>{cells}</tr>")
    return (
        "<table style='border-collapse:collapse;border:1px solid #999'>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{''.join(body_rows)}</tbody></table>"
    )

def display_table(columns, rows, title=None):
    if title:
        print(title)
        print("-" * len(title))
    if in_ipython_frontend() and display is not None and HTML is not None:
        display(HTML(table_to_html(columns, rows)))
    else:
        print(table_to_text(columns, rows))

def show_markdown(text):
    if in_ipython_frontend() and display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)

size_to_entry = {tuple(entry['size']): entry for entry in orthogonal_results}

## Summary table: main orthogonal trials and non-orthogonal control

The table below focuses on the statistics actually returned by the script. The
non-orthogonal control is reported separately so that support for orthogonal
conjugation is not conflated with arbitrary linear changes of basis.

In [ ]:
summary_rows = []
for entry in orthogonal_results:
    summary = entry['summary']
    summary_rows.append({
        'Condition': f"Orthogonal {entry['size_label']}",
        'Trials': entry['trial_count'],
        'Mean': sci(summary['mean']),
        'Median': sci(summary['median']),
        'Std': sci(summary['std']),
        'Max': sci(summary['max']),
        '95th pct': sci(summary['q95']),
        '#(<1e-12)': entry['threshold_counts']['1e-12'],
    })

agg = orthogonal_aggregate['summary']
summary_rows.append({
    'Condition': 'Orthogonal aggregate',
    'Trials': orthogonal_aggregate['trial_count'],
    'Mean': sci(agg['mean']),
    'Median': sci(agg['median']),
    'Std': sci(agg['std']),
    'Max': sci(agg['max']),
    '95th pct': sci(agg['q95']),
    '#(<1e-12)': orthogonal_aggregate['threshold_counts']['1e-12'],
})

control_summary = nonorthogonal_control['summary']
summary_rows.append({
    'Condition': f"Non-orthogonal control {nonorthogonal_control['size_label']}",
    'Trials': nonorthogonal_control['trial_count'],
    'Mean': sci(control_summary['mean']),
    'Median': sci(control_summary['median']),
    'Std': sci(control_summary['std']),
    'Max': sci(control_summary['max']),
    '95th pct': sci(control_summary['q95']),
    '#(<1e-12)': nonorthogonal_control['threshold_counts']['1e-12'],
})

display_table(
    columns=['Condition', 'Trials', 'Mean', 'Median', 'Std', 'Max', '95th pct', '#(<1e-12)'],
    rows=summary_rows,
    title='Main experiment summary'
)

In [ ]:
orth_mean = orthogonal_aggregate['summary']['mean']
orth_max = orthogonal_aggregate['summary']['max']
control_mean = nonorthogonal_control['summary']['mean']
ratio = control_mean / max(orth_mean, 1e-300)

show_markdown(f"""**Interpretation.** Across the sampled orthogonal trials, the aggregate
mean relative error is `{orth_mean:.2e}` and the aggregate max error is
`{orth_max:.2e}`, i.e. at float64 roundoff scale. By contrast, the
non-orthogonal control has mean error `{control_mean:.2e}`. In this run the
non-orthogonal mean is about `{ratio:.2e}` times larger than the orthogonal
mean. This supports the narrow claim that the implemented update is
numerically consistent with orthogonal conjugation, while arbitrary
non-orthogonal transforms do not preserve the identity.""")


## Orthogonal error distributions by tested size

The next figure shows the log-scale distribution of relative errors for the
two sampled orthogonal cases (`4x4` and `8x8`).

In [ ]:
bins = config['log10_histogram_bins']
fig, axes = plt.subplots(1, len(orthogonal_results), figsize=(12, 4), sharey=True)
if len(orthogonal_results) == 1:
    axes = [axes]

for ax, entry in zip(axes, orthogonal_results):
    log_errors = np.log10(entry['errors'] + 1e-20)
    ax.hist(log_errors, bins=bins, color='#4c72b0', edgecolor='black', alpha=0.85)
    ax.set_title(
        f"{entry['size_label']}\nmean={entry['summary']['mean']:.2e}, max={entry['summary']['max']:.2e}"
    )
    ax.set_xlabel('log10 relative error')
    ax.set_ylabel('trial count')
    ax.grid(alpha=0.25)

fig.suptitle('Sampled orthogonal-conjugation error distributions by size', y=1.03)
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
orthogonal_maxima_text = ", ".join(
    f"`{entry['size_label']}` max = `{entry['summary']['max']:.2e}`"
    for entry in orthogonal_results
)

show_markdown(f"""**Interpretation.** For the sampled orthogonal trials, all reported
errors lie far below the experiment's `1e-12` pass threshold. The observed
per-size maxima are {orthogonal_maxima_text}. This is consistent with the
implemented single-step identity up to floating-point roundoff on the tested
sizes. It is evidence for the sampled toy setting, not a proof for all sizes
or all orthogonal conjugators.""")


## Orthogonal vs non-orthogonal comparison

A central control question is whether low error is specific to orthogonal
conjugation or merely a generic numerical accident. The figure below compares
the sampled orthogonal trials to the non-orthogonal control.

In [ ]:
comparison_data = [entry['errors'] for entry in orthogonal_results] + [
    nonorthogonal_control['errors']
]
labels = [f"Orthogonal {entry['size_label']}" for entry in orthogonal_results] + [
    f"Non-orthogonal control {nonorthogonal_control['size_label']}"
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(comparison_data, tick_labels=labels, showfliers=True)
ax.set_yscale('log')
ax.axhline(1e-12, color='tab:red', linestyle='--', linewidth=1.5, label='1e-12 pass threshold')
ax.set_ylabel('relative error (log scale)')
ax.set_title('Orthogonal trials vs non-orthogonal control')
ax.grid(alpha=0.25, which='both')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
show_markdown(f"""**Interpretation.** The orthogonal cases remain clustered at
approximately machine precision, whereas the non-orthogonal control is
centered near `{nonorthogonal_control['summary']['mean']:.2e}`. The control
therefore fails by orders of magnitude relative to the orthogonal cases, but
the observed scale is about `1e-2`, not `1e-1` or larger. The notebook and
script are now calibrated to that actual observed threshold behavior.""")


## Sensitivity to finite Newton-Schulz iteration count

The script also sweeps the number of Newton-Schulz iterations over
`{1, 3, 5, 10, 20}` while keeping the rest of the toy experiment fixed at
`4x4`. This tests whether conjugation equivariance appears tied to a specific
iteration count or is stable across the sampled finite-iteration map.

In [ ]:
ns_rows = []
for entry in ns_sensitivity:
    summary = entry['summary']
    ns_rows.append({
        'NS iters': entry['ns_iters'],
        'Seed': entry['rng_seed'],
        'Trials': entry['trial_count'],
        'Mean': sci(summary['mean']),
        'Median': sci(summary['median']),
        'Std': sci(summary['std']),
        'Max': sci(summary['max']),
        '#(<1e-12)': entry['threshold_counts']['1e-12'],
    })

display_table(
    columns=['NS iters', 'Seed', 'Trials', 'Mean', 'Median', 'Std', 'Max', '#(<1e-12)'],
    rows=ns_rows,
    title='Finite Newton-Schulz sensitivity summary'
)

In [ ]:
ns_iters = [entry['ns_iters'] for entry in ns_sensitivity]
ns_means = [entry['summary']['mean'] for entry in ns_sensitivity]
ns_maxes = [entry['summary']['max'] for entry in ns_sensitivity]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ns_iters, ns_means, marker='o', linewidth=2, label='mean error')
ax.plot(ns_iters, ns_maxes, marker='s', linewidth=2, label='max error')
ax.set_yscale('log')
ax.set_xlabel('Newton-Schulz iterations')
ax.set_ylabel('relative error (log scale)')
ax.set_title('Equivariance error vs finite Newton-Schulz iteration count')
ax.grid(alpha=0.25, which='both')
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
show_markdown(f"""**Interpretation.** Across the sampled sensitivity sweep, the mean
errors range from `{min(ns_means):.2e}` to `{max(ns_means):.2e}` and the max
errors range from `{min(ns_maxes):.2e}` to `{max(ns_maxes):.2e}`. In this toy
setting, sampled orthogonal-conjugation equivariance remains at roundoff
scale across all tested iteration counts. This should be interpreted as
evidence about the finite conjugation-equivariant map being used, not as a
statement that Newton-Schulz has converged to the exact polar factor in each
case.""")


## H1-H4 outcomes returned by the script

These are the same thresholded checks used by the script's console report. The
notebook reports them directly rather than re-deriving a different set of
criteria.

In [ ]:
hypothesis_rows = []
for label in ['H1', 'H2', 'H3', 'H4']:
    entry = hypotheses[label]
    hypothesis_rows.append({
        'Hypothesis': label,
        'Statement': entry['statement'],
        'Criterion': entry['criterion_text'],
        'Observed': entry['observed_text'],
        'Outcome': 'PASS' if entry['passed'] else 'FAIL',
    })

display_table(
    columns=['Hypothesis', 'Statement', 'Criterion', 'Observed', 'Outcome'],
    rows=hypothesis_rows,
    title='H1-H4 results'
)

print()
print(f"Tests passed: {verdict['passed']}/{verdict['total']}")
print(f"Headline    : {verdict['headline']}")

## Scope and limitations

What this notebook **does test**:

- zero-momentum, one-step Muon update
- finite Newton-Schulz iterate used in the script
- float64 CPU arithmetic
- sampled orthogonal conjugations at `4x4` and `8x8`
- a non-orthogonal control and an iteration-count sensitivity sweep

What this notebook **does not test**:

- all orthogonal matrices or all matrix sizes
- momentum-buffer dynamics
- long optimization trajectories
- convergence quality of finite Newton-Schulz to the exact polar factor
- comparisons against SGD, Adam, or other optimizers

## Calibrated conclusion

The conclusion should match the implemented evidence exactly: the experiment is
a small, fast, reproducible numerical check showing that the script's
zero-momentum single-step Muon update is *numerically consistent* with
orthogonal-conjugation equivariance on the sampled toy cases, while the
non-orthogonal control does not satisfy the same identity.

In [ ]:
orthogonal_summary_lines = [
    (
        f"- Sampled orthogonal `{entry['size_label']}` mean/max = "
        f"`{entry['summary']['mean']:.2e}` / `{entry['summary']['max']:.2e}`."
    )
    for entry in orthogonal_results
]
orthogonal_summary_block = '\n'.join(orthogonal_summary_lines)

show_markdown(f"""### Final evidence-based summary

{orthogonal_summary_block}
- Aggregate orthogonal mean/max = `{orthogonal_aggregate['summary']['mean']:.2e}` / `{orthogonal_aggregate['summary']['max']:.2e}`.
- Non-orthogonal control mean/max = `{nonorthogonal_control['summary']['mean']:.2e}` / `{nonorthogonal_control['summary']['max']:.2e}`.
- H1-H4 outcome = `{verdict['passed']}/{verdict['total']}` checks passed.

**Conclusion.** Within this toy setup, the sampled numerical evidence is fully
consistent with exact orthogonal-conjugation equivariance of the implemented
zero-momentum single Muon step up to float64 roundoff. The evidence is strong
for the tested cases, but it should still be described as a sampled numerical
verification rather than a universal proof.""")